# Data Processing

The data we have used for our project came from Understat, however before it could be used we had to ensure it was in the correct form so that all of our models would work without error. The code for how this data was downloaded can be seen in Dylan's folder.

This first chunk of code imports the raw dataset directly from the project's GitHub repository into this notebook.


In [1]:
# Load raw data directly from GitHub repository
import pandas as pd

url = "https://raw.githubusercontent.com/donny31415/Group-Sprint-1/main/data/understat_shots.csv"
df_raw = pd.read_csv(url, keep_default_na=False, na_values=["NA"])

print(df_raw.shape)
df_raw.head()

## Inspecting the Data
The following code was used to inspect the structure of the dataset, identify missing values and check for duplicate observations. The initial inspection showed that there were 0 duplicate records. The x-coordinate ranges from 0 to 120, while the y-coordinate ranges from 0 to 80, with the goal located at x = 120.

The dataset also contains 212 own goals, 494 penalties and data covering six seasons. However, the most recent season (2026/27) contains only 1,665 shots, which is considerably less data than may be available for the other seasons. As a consequence of this, all of the models removed this season since it wasn't complete.


In [2]:
df_raw.info()
summary = pd.DataFrame({
    "dtype": df_raw.dtypes,
    "n_missing": df_raw.isna().sum(),
    "n_unique": df_raw.nunique(),
})
display(summary)
print("Duplicate rows:", df_raw.duplicated().sum())
print("Duplicate shot_id:", df_raw["shot_id"].duplicated().sum())
display(df_raw[["x", "y", "minute", "xg", "angle_to_goal"]].describe())

for c in ["season", "result", "situation", "shot_type", "last_action"]:
    print(f"\n{c}\n", df_raw[c].value_counts(dropna=False))

## Feature Removals and Data Preprocessing
Several features and observations were removed or transformed to ensure the dataset was suitable for training and evaluating the four predictive models.

The target variable was defined as whether result == "Goal". Both result and xg were excluded from the feature matrix. This avoids data leakage, as xg is Understat's own expected goals model output. Instead, xG was retained separately as a benchmark against which the four models could be compared.

The data was split chronologically by season rather than randomly. This helps prevent shots from the same matches appearing in both the training and testing sets, which could lead to overly optimistic performance estimates. The complete seasons from 2021/22 to 2024/25 were used for training, with 2025/26 reserved for testing. The partial 2026/27 season was retained as a separate held-out check.

A total of 212 own goals were removed, leaving 50,304 shots for analysis. Own goals were excluded because they are not genuine shot attempts and have an xG value of zero.

Additional preprocessing steps included:
- Replacing missing last_action values with "None".
- Converting is_home and is_assisted into binary 1/0 variables.
- Trimming whitespace from text columns.
- Checking that xG values fall between 0 and 1.
- Ensuring pitch coordinates are within the expected ranges.
- Checking that minute falls between 0 and 130.
- Verifying that score_diff correctly represents the difference between the team's and opponent's goals before each shot.

These preprocessing decisions aim to ensure that the models learn from genuine shot characteristics and match circumstances, rather than information that directly reveals the outcome.

In [3]:
df = df_raw.copy()

# 1. Own goals are not shot attempts at goal (xg = 0, positioned at the scorer's
#    own end). Their effect on the score is already in team_goals_before /
#    opp_goals_before for later shots.
df = df[df["result"] != "OwnGoal"]

# 2. Penalties are now KEPT so that is_penalty carries information.
DROP_PENALTIES = False
if DROP_PENALTIES:
    df = df[~df["is_penalty"]]

df = df.copy()  # avoids SettingWithCopy warnings

# 3. last_action: "None" is Understat's label for no recorded prior action.
df["last_action"] = df["last_action"].fillna("None")

# 4. Boolean -> 1/0
BINARY_COLS = [
    "is_home", "is_assisted",
    "is_penalty", "is_open_play", "is_set_piece", "from_corner", "is_free_kick",
]
for c in BINARY_COLS:
    df[c] = df[c].astype(int)

for c in ["team", "season", "shot_type", "last_action", "result"]:
    df[c] = df[c].astype("string").str.strip()

# 5. Sanity checks
assert df["xg"].between(0, 1).all()
assert df["x"].between(0, 120).all() and df["y"].between(0, 80).all()
assert df["minute"].between(0, 130).all()
assert (df["score_diff"] == df["team_goals_before"] - df["opp_goals_before"]).all()

assert (df["is_open_play"] + df["is_set_piece"] + df["is_penalty"] == 1).all()
assert (df["from_corner"] <= df["is_set_piece"]).all()
assert (df["is_free_kick"] <= df["is_set_piece"]).all()

print(df.shape)

## One-Hot Encoding
One-hot encoding was applied to the categorical variables shot_type and last_action to convert them into numerical features suitable for the machine learning models.

The shot_type variable contains four distinct categories: Head, LeftFoot, OtherBodyPart and RightFoot. Each category was converted into its own binary (1/0) column, indicating whether a particular shot type applies to an observation. An additional check was included to ensure that every shot belongs to exactly one category.

As these categories are mutually exclusive and collectively exhaustive, the resulting columns are redundant if all four are included. Therefore, one category must be dropped when fitting the unpenalised logistic regression model to avoid perfect multicollinearity. This category acts as the reference or baseline against which the other coefficients are interpreted.

In [4]:
import re

# CamelCase -> snake_case, e.g. RightFoot -> right_foot
def to_snake(s):
    return re.sub(r"(?<!^)(?=[A-Z])", "_", s).lower()

print(df["shot_type"].value_counts(dropna=False))
shot_types = sorted(df["shot_type"].dropna().unique())

shot_type_cols = []
for st in shot_types:
    col = to_snake(st)
    df[col] = (df["shot_type"] == st).astype(int)
    shot_type_cols.append(col)

print("\nNew columns:", shot_type_cols)

# Check: every shot is exactly one type
assert (df[shot_type_cols].sum(axis=1) == 1).all()
display(df[["shot_type"] + shot_type_cols].head())
assert (df["is_header"].astype(int) == df["head"]).all()

The last_action variable contains a larger number of categories. To avoid creating too many sparse features, only categories occurring in at least 100 shots were retained. This resulted in 16 individual binary indicator columns, while the 18 less frequent categories were assigned zero across all retained indicators. The "None" category was retained as a separate feature, accounting for 6,177 shots.


In [5]:
MIN_OBS = 100

la_counts = df["last_action"].value_counts(dropna=False)
print(la_counts)

keep_vals = sorted(la_counts[la_counts >= MIN_OBS].index)
dropped_vals = sorted(la_counts[la_counts < MIN_OBS].index)
print(f"\nKept ({len(keep_vals)}):", keep_vals)
print(f"Below {MIN_OBS} ({len(dropped_vals)}):", dropped_vals)

last_action_cols = []
for val in keep_vals:
    col = "last_action_" + to_snake(val)
    df[col] = (df["last_action"] == val).astype(int)
    last_action_cols.append(col)

assert (df[last_action_cols].sum(axis=1) <= 1).all()
n_none = (df[last_action_cols].sum(axis=1) == 0).sum()
print(f"\nShots with all flags 0 (rare last_action values): {n_none}")
display(df[["last_action"] + last_action_cols].head())

One-hot encoding is necessary because logistic regression (both standard and penalised) and random forest models require numerical inputs. It also makes the categorical information explicit, allowing the logistic regression coefficients to be interpreted in relation to a reference category. The minimum frequency threshold of 100 observations represents a bias-variance trade-off. Retaining very rare categories can produce unstable coefficient estimates and features that are poorly represented in individual seasonal splits. Grouping these rare categories implicitly into the baseline reduces sparsity while retaining the most informative actions. The choice of threshold could be investigated through sensitivity analysis, comparing alternative minimum frequencies of 50 and 200 observations to assess whether the modelling results are affected by the treatment of rare categories.

## Restructuring Features 

In our model, we are not concerned with how a shot failed to result in a goal; instead, we are only interested in whether a goal was scored or not. Therefore, in this section of code, the result variable is converted into a binary format. Previously, it contained five different outcomes describing the result of each shot. These are simplified into two categories, with goals assigned a value of 1 and all other outcomes assigned a value of 0. This allows the models to focus on predicting the probability of a shot resulting in a goal.

In [6]:
assert set(df["result"].unique()) <= {"Goal", "MissedShots", "BlockedShot", "SavedShot", "ShotOnPost"}, \
    "result already converted; rerun from the top"

print(df["result"].value_counts())
df["result"] = (df["result"] == "Goal").astype(int)

print("\nAfter conversion:")
print(df["result"].value_counts())
print("Goal rate:", df["result"].mean().round(4))

## Final Complete Dataset

This code shows which features were kept and does a final check to ensure that we have no missingness.

In [7]:
KEEP_COLS = [
    "is_home", "team", "season",
    "xg", "minute", "result",
    "x", "y", "angle_to_goal",
    "is_assisted",
    "is_penalty", "is_open_play", "is_set_piece", "from_corner", "is_free_kick",
    "team_goals_before", "opp_goals_before", "score_diff",
] + shot_type_cols + last_action_cols

df_clean = df[KEEP_COLS].reset_index(drop=True)

print("Dropped columns:", sorted(set(df_raw.columns) - set(KEEP_COLS)))
print(df_clean.shape)
print(df_clean.isna().sum().sum(), "missing values remaining")
assert set(df_clean["result"].unique()) == {0, 1}
display(df_clean.head())
# Note: Output saving code removed as requested.